In [1]:
import sys
import requests
sys.path.append(r'E:\wangzhilin\QuantumGalaxy')
sys.path.append(r'E:\wangzhilin\QuantumGalaxy\server_new\blueprints')
from live_editor.models import bitable_works
import json
from QGI.feishu import base_fetch_func,fetch_tenant_token
from copy import deepcopy
bitapp_auth={'app_id':'cli_a1fc0a192db8100c','app_secret':'nNk1fT8h2eVi8B2ymIFangHTpYrOAJTD'}
import pandas as pd
def build_headers():
    
    token=fetch_tenant_token(bitapp_auth)
    #token='u-3atGsodiN9391_BlvmfeZh1kkwt5hlQxo0G0hgew2Ijc'
    headers = {
        'Authorization':
        'Bearer ' + token,  # your access token
        #'Content-Type': 'application/json; charset=utf-8'
    }
    return headers


In [2]:
import logging
def get_logger(name):
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    fh = logging.FileHandler(
        r"E:\wangzhilin\QuantumGalaxy\opt\combine_check\log\log.log",
        'a',
        encoding='utf-8')
    fh.setLevel(logging.INFO)
    ch = logging.StreamHandler()
    ch.setLevel(logging.DEBUG)
    formatter = logging.Formatter(
        #fmt='%(asctime)s %(name)-12s %(levelname)-8s %(message)s',
        fmt='%(levelname)-8s %(message)s',
        datefmt='%m-%d %H:%M')
    ch.setFormatter(formatter)
    fh.setFormatter(formatter)
    logger.addHandler(ch)
    logger.addHandler(fh)
    return logger
logger=get_logger('combine_check')

In [3]:
logger.warning('start')

WARNING  start


#### 将2个（多个？）骨干多维表格中的外骨干节点和关系抽出，进行对照，排除冲突和错误

规则：
1. 表格内外部节点/边是否都有外部骨干名字，外部骨干名字（只有写了外部骨干才是外部节点）
2. 表格内外部节点和边是否意义对应（表格内有外部节点的话，则至少有1条外部边从内部节点或其他外部节点连向该外部节点），存在不对应的点和边则加入error，该表不联查 另，外部节点连向外部节点的边和节点都warning，不参与检查和报错

3. 满足以上两条规则的表格可以参与联合检查，每张表每张表排除自身error和warning后生成n组外边（主主体关系客体）及其外骨干名
对m张表的外边匹配相应外骨干的外边，如果匹配失败则加入该表的错误外边，成功的一对外边成为星图边界

4. 对各个粗骨干的内部节点命名进行唯一性检查，如果重复则是可能的外部节点
4. 最终生成匹配成功的星图边界表和其他表的错误报告（节点和边不对应error，外边匹配失败error，多层2边warning）
5. 画图（原本的内容+正确的外点外边，但是点和边有重复,解决冲突后用cypher可以录入）

In [4]:
def append_df(df,new_row):
    '''["level","body","backbone",""rough"]'''
    df=pd.concat((df,pd.DataFrame(data=[new_row],columns=['level','body','backbone','rough'])))
    return df
def get_mark(backbone_mark):
    if backbone_mark:
        mark=[]
        
        for record in backbone_mark:
            fields=record['fields']
            #print(fields)
            if fields['本骨干标记']:
                mark.append(fields['骨干名'])
        if mark:
            if len(mark)==1:
                return (True,mark[0])

            else:
                return (False,'找到复数勾选骨干：%s'%mark)
        else:
            return (False,'没有勾选骨干标记')
    else:
        return (False,'没有勾选本骨干')


In [5]:
def get_files(fold_token):
    logger.info('开始检查文件，文件夹地址为　https://quantumgalaxy.feishu.cn/drive/folder/%s'%fold_token)
    r0=requests.get('http://82.156.248.152:81/fs/fold_meta?token=%s'%fold_token).json()
    
    r1=requests.get('http://82.156.248.152:81/fs/fold_children?fld_token=%s'%fold_token).json()
    if r0['code']==0 and r1['code']==0:
        files=r1['data']['files']
        data={}
        for file in files:
            if file['name'] not in ['电力储能骨干核心表Demo','xx骨干核心表模板及范例']:
            #if file['type']=='bitable' and '核心表' in file['name'] and file['name']!='xx骨干核心表Demo':
            #if True:
                data[file['name']]=file['token']
            else:
                logger.warning('排除该文件，类型:%s,名称:%s'%(file['type'],file['name']))
        num=len(data)
        
        logger.info('检查文件夹成功，共查到%d张多维表格参与联查，如下：'%num)
        for d in data.keys():
            logger.info(d)
        
        return {'code':0,'data':data}
    else:
        logger.info('检查文件夹失败，检查地址')
        return {'code':r0['code'],'data':r0}


In [6]:
def external_check(internal_result,files,report_df):
    '''对所有表格中externallinks+backbonemark进行检查，其对应的
    external_link结构：  
    'external_links': [{'bitapp_id': 'recNx4LYVO',
    'source': 'reclwpDIsc',
    'target': 'recDC1O4Rf',
    'category': '父节点',
    'checked': True,
    'source_name': '精煤',
    'target_name': '动力煤',
    'external_backbone_name': '电力',
    'external_backbone': '未知'},
    
    A表 有externallink L1,其external_backbone为B，则B表必须有相同sourcenametargaetname的externallink L1，其externalbackbone为A。成功则L1 l2为正确边界(只判断L1在出报告更方便？），否则暴露L1为错误'''
    result=deepcopy(internal_result)
    for token,data in result.items():
        logger.critical('对标记为"%s"的表格的边界进行检查,该表共有%s条外边参与联调'%(data['backbone_mark'],len(data['external_links'])))
        logger.critical(data['backbone_mark'])
        data['error_external_links']=[]
        data['good_external_links']=[]
        #print(data['external_links'])
        for link in data['external_links']:
            linktext=link['source_name']+'--'+link['category']+'->'+link['target_name']
            other_token=link['external_backbone']
            #print(internal_result[other_token])
            #othertoken是在internalcheck时根据file表查到
            get=False
            if other_token in internal_result.keys():
                for other_link in internal_result[other_token]['external_links']:
                #当表格在files里，但权限有问题，该表的token不会出现在internalresult中，需要提示
                    otherlinktext=other_link['source_name']+'--'+other_link['category']+'->'+other_link['target_name']
                    if otherlinktext==linktext:
                        get=True
                        if other_link['external_backbone']==token:
                            try:
                                if link.get('target'):
                                    link['source']=other_link['source']
                                else:
                                    print(link)
                                    print(other_link)
                                    link['target']=other_link['target']
                                data['good_external_links'].append(link)
                                logger.info('星图边界识别成功，在"%s"骨干中查到对应边："%s"'%(link['external_backbone_name'],linktext))
                            except:
                               data['error_external_links'].append(link) 
                               body='找到文本相同的星图边界但起终点有问题，可能是方向写反了或者外骨干标重了，如无法解决联系王之霖。边信息：%s'%linktext
                               logger.error(body)
                               report_df=append_df(report_df,['error',body,data['backbone_mark'],True])
                        else:
                            
                            data['error_external_links'].append(link)
                            body='星图边界识别到对应的边，但对应边的外骨干标记与本骨干不符，对应边的外骨干标记为："%s"，对应边为："%s"'%(link['external_backbone_name'],linktext)
                            logger.error(body)
                            report_df=append_df(report_df,['error',body,data['backbone_mark'],True])
                        break
            else:
                data['error_external_links'].append(link)
                body='星图边界识别失败，该边："%s"所对应的"%s"骨干没有经过内部检查，这是对应骨干而非本边的问题'%(linktext,link['external_backbone_name'])
                logger.error(body)
                report_df=append_df(report_df,['error',body,data['backbone_mark'],True])
            if not get:
                data['error_external_links'].append(link)
                body='星图边界识别失败，该边："%s"在对应"%s"骨干中没找到对应边界'%(linktext,link['external_backbone_name'])
                logger.error(body)
                report_df=append_df(report_df,['error',body,data['backbone_mark'],True])
        logger.critical('完成对标记为"%s"的表格的边界进行检查,该表共有%s条外边参与联调，通过边%s条，冲突边%s条\n'%(data['backbone_mark'],len(data['external_links']),len(data['good_external_links']),len(data['error_external_links'])))
        #print(data['backbone_mark'])
        data.pop('external_links')
    return result,report_df


In [7]:
def internal_check(data1,files,report_df):
    '''对多维表格data进行内部检查，梳理出所有外部节点和其对应边，其中孤立外部节点加入error，多层外部节点加入warning，其他加入external,并把原data中其他数据分类到
    'internal_links':[],'external_links':[],'warning_links':[],'error_nodes':[],'internal_nodes':[],'external_nodes':[],
    加上骨干标记'''
    data=deepcopy(data1)
    result={'internal_links':[],'external_links':[],'warning_links':[],'error_links':[],'error_nodes':[],'internal_nodes':[],'external_nodes':[]}
    r,mark=get_mark(data['backbone_mark'])
    if r:
        logger.critical('本骨干标记：')
        logger.critical(mark)
        result['backbone_mark']=mark
    else:
        body='检查到骨干定义表问题：%s'%mark
        logger.error(body)
        report_df=append_df(report_df,['error',body,mark,True])
        return None,None
    
    #result.update(data)
    for link in data['links']:
        if not link['category'] in ['原材料','储运','父节点','供应','同位','生产','反映']:
            link['error']=True
            result['error_links'].append(link)
            body='该边的关系有误：“%s”'%link['category']
            logger.error(body)
            report_df=append_df(report_df,['error',body,mark,True])
        for o in data['nodes']:
            if o['bitapp_id']==link['target']:
                link['target_name']=o['name']
            if o['bitapp_id']==link['source']:
                link['source_name']=o['name']
        if not (link.get('target_name') and link.get('source_name')):
            link['error']=True
            result['error_links'].append(link)
            body='出现边没有找到对应的起/终点，可能是因为起终点没有勾选粗骨干权限。id:"%s",起点："%s" 终点:"%s"'%(link.get('bitapp_id'),link.get('source_name_ref','unknown'),link.get('target_name_ref','unknown'))
            logger.error(body)
            report_df=append_df(report_df,['error',body,mark,True])
            
    for node in data['nodes']:
        if not node['more_info'].get('external_backbone'):
            result['internal_nodes'].append(node)
        else:
            
            id=node['bitapp_id']
            backbone=node['more_info'].get('external_backbone')

            if backbone:
                backbone=backbone[0]
            check=0
            if not files.get(backbone,None):
                '''该外部节点的‘所属外部骨干’字段与文件目录files比对结果，因此要求字段内骨干名与files名一致，不然认为没有找到外骨干'''
                '''涉及errornode的边移入errorlink'''
                logger.error('“%s”节点没有找到对应的外部骨干：“%s”'%(node['name'],backbone))
                #error
                result['error_nodes'].append(node)
                for link in data['links']:
                    if link['source']==id or link['target']=='id':
                        if link['source']==id :
                            other=link['target']
                            for o in data['nodes']:
                                if o['bitapp_id']==other :
                                    #link['source_name']=node['name']
                                    #link['target_name']=o['name']
                                    link['external_backbone_name']=backbone
                                    link['external_backbone']=files.get(backbone,'未知')
                        else:
                            other=link['source']
                            for o in data['nodes']:
                                if o['bitapp_id']==other :
                                    #link['source_name']=o['name']
                                    #link['target_name']=node['name']
                                    link['external_backbone_name']=backbone
                                    link['external_backbone']=files.get(backbone,'未知')
                        result['error_links'].append(link)
                        body='因“%s”节点没有找到对应外部骨干，与其相连的边也出错,不参与联调：[%s]-(%s)->[%s]'%(node['name'],link['source_name'],link['category'],link['target_name'])
                        logger.error(body)
                        report_df=append_df(report_df,['error',body,mark,True])
                        link['error']=True
            else:
                result['external_nodes'].append(node)

                for link in data['links']:
                    
                        if link['source']==id :
                            
                            #从data中除去该边，只留纯外边
                            other=link['target']
                            for o in data['nodes']:
                                if o['bitapp_id']==other :
                                        
                                        
                                        link['source_name']=node['name']
                                        link['target_name']=o['name']
                                        link['external_backbone_name']=backbone
                                        link['external_backbone']=files.get(backbone,'未知')
                                        if not (o['more_info'].get('external_backbone') or o['category']=='指标'):
                                            #说明另一点为内部
                                            link1=deepcopy(link)
                                            link1.pop('source')
                                            logger.info('检查到与“%s”骨干间的正确星图边界：[%s]-(%s)->[%s]'%(backbone,link['source_name'],link['category'],link['target_name']))
                                            result['external_links'].append(link1)
                                            check=max(check,2)
                                            
                                        else:
                                        
                                            if not link.get('checked'):
                                                result['warning_links'].append(link)
                                                #说明另一点为外部,该边加入warning
                                                body='检查到本骨干之外的,属于“%s”骨干的边，该边不参与冲突检查：[%s]-(%s)->[%s]'%(backbone,link['source_name'],link['category'],link['target_name'])
                                                logger.warning(body)
                                                report_df=append_df(report_df,['warning',body,mark,True])
                                            check=max(check,1)
                                        continue
                            link['checked']=True

                                
                                
                        elif link['target']==id:
                            
                            other=link['source']
                            for o in data['nodes']:
                                if o['bitapp_id']==other:
                                        
                                        
                                        link['source_name']=o['name']
                                        link['target_name']=node['name']
                                        link['external_backbone_name']=backbone
                                        link['external_backbone']=files.get(backbone,'未知')
                                        if not (o['more_info'].get('external_backbone') or o['category']=='指标'):
                                            #说明另一点为内部,该边加入external
                                            link1=deepcopy(link)
                                            link1.pop('target')
                                            logger.info('检查到与“%s”骨干间的正确星图边界：[%s]-(%s)->[%s]'%(backbone,link['source_name'],link['category'],link['target_name']))
                                            result['external_links'].append(link1)
                                            check=max(check,2)
                                            
                                        else:
                                            
                                            if not link.get('checked'):
                                                
                                                result['warning_links'].append(link)
                                                #说明另一点为外部,该边加入warning
                                                body='检查到本骨干之外的,属于“%s”骨干的边，该边不参与冲突检查：[%s]-(%s)->[%s]'%(backbone,link['source_name'],link['category'],link['target_name'])
                                                logger.warning(body)
                                                report_df=append_df(report_df,['warning',body,mark,True])
                                            check=max(check,1)
                                        continue
                            link['checked']=True
                        else:
                            #result['internal_links'].append(link)
                            pass
                            '''所有边均未触发则error 该外部节点为孤立，未连入当前骨干'''
                        
                if check==2:
                    #ok
                    pass
                elif check==1:
                    #warning
                    pass
                else:
                    body='检查到孤立外骨干节点，该点为内部错误，不参与冲突检查：%s'%node['name']
                    logger.error(body)
                    report_df=append_df(report_df,['error',body,mark,True])
                    #error
                    result['error_nodes'].append(node)
                    pass
    for link in data['links']:
                if not link.get('checked'):
                    if not link.get('error'):
                        #没有check表示起终点均不是外部
                        result['internal_links'].append(link)
    return result,report_df
#data1=deepcopy(data)
#internal_check(data1)
            

In [8]:
testfiles={'第一个高级权限设置和外骨干范例': 'bascnzsAjUXwr4gUkaIpqi8BBaf','第二个高级权限设置和外骨干范例':'bascn9HmmVcWhW7fYfQdpJZXtdh'}

In [9]:
def find_common_elements(result,files):
    #检查所有表格里 internal_nodes 的name是否有重复，有的话给出表格token和节点name
    dictionary={}
    for k,v in result.items():
        names=[]
        #if v.get('good_external_nodes'):
        for item in v['internal_nodes']:
        #for item in v['good_external_nodes']:
            if item['category']!='企业':
                names.append(item['name'])
        dictionary[k]=names
    
    common_elements = []
    for key1, lst1 in dictionary.items():
        for key2, lst2 in dictionary.items():
            if key1 != key2:
                for element in lst1:
                    if element in lst2  and not ([element,(key1,key2)]  in common_elements or ([element,(key2,key1)]  in common_elements)):
                        common_elements.append([element,(key1,key2)])
    for i in common_elements:
        tokens=i.pop(1)
        names=({v:k for k,v in files.items()}[tokens[0]],{v:k for k,v in files.items()}[tokens[1]])
        i.append(names)

    return common_elements

In [10]:
def work1(files,report_df):
    logger.info('start')

    
    
    raw_datas={}
    internals={}
    externals={}
    internal_result={}
    errorfile=[]
    #result={'raw':raw_datas,'internal':internals,'external':externals}
    for name,token in files.items():
        '''file:{name,token}'''

        #data=bitable_works(token,subgraph_name='粗骨干图')
        data=bitable_works(token)
        raw_datas[token]=data
        logger.critical('对 “%s”  多维表格进行内部检查'%name)
        if data.get('code'):
            body='对 “%s”  多维表格进行内部检查失败，错误信息：%s\n'%(name,data)
            logger.critical(body)
            report_df=append_df(report_df,['critical',body,name,True])
            errorfile.append(name)
            continue
        r,report_df=internal_check(data,files,report_df)
        if r:
            internal_result[token]=r
            logger.critical('完成对 “%s”  多维表格进行内部检查\n'%name)
        else:
            errorfile.append(name)
            logger.critical('无法对 “%s”  多维表格进行内部检查\n'%name)
    logger.critical('内部检查完成，以下表格出现问题不参与联调:%s\n'%errorfile)
    #for errorname in errorfile:
        #files.pop(errorname)
    return internal_result,report_df
def work2(internal_result,files,report_df):

    logger.critical('开始外部联调，参与表格为：')
    logger.critical(str(list(files.keys()))+'\n')
    
    result,report_df=external_check(internal_result,files,report_df)
    
    '''外部检查'''
        
    return result,report_df
    

In [11]:

files={'区域':'DARKb6rCUaV2OqstRYdcyANBnbc'}

In [12]:
r=requests.get(
            r'http://172.21.0.14:81/bitapp/tables?app_token=%s' % 'DARKb6rCUaV2OqstRYdcyANBnbc')
r.json()

{'code': 0,
 'data': [{'name': '节点表', 'revision': 10, 'table_id': 'tbl0X22l77e57rYz'},
  {'name': '关系表', 'revision': 8, 'table_id': 'tblmKxDCwLyVBzw8'},
  {'name': '监测指标表', 'revision': 2, 'table_id': 'tblhsMhLWddJG5xZ'},
  {'name': '局部图列表', 'revision': 0, 'table_id': 'tblkHxVIKJnRJwih'},
  {'name': '世界线跟踪表', 'revision': 0, 'table_id': 'tblGeDtlfBNPGWg9'},
  {'name': '关系定义表', 'revision': 1, 'table_id': 'tblibBqeLKEIID7r'},
  {'name': '骨干定义表', 'revision': 6, 'table_id': 'tbltGzurJw2WV4YD'},
  {'name': '画图草稿', 'revision': 3, 'table_id': 'tblx8OCevTMu1jYx'}]}

In [13]:
logger.info('start')

INFO     start


In [14]:
fold='fldcn2VlYvt8Efb9PqZwO51GWud'
files=get_files(fold)
if files['code']==0:
        files=files['data']
files


INFO     开始检查文件，文件夹地址为　https://quantumgalaxy.feishu.cn/drive/folder/fldcn2VlYvt8Efb9PqZwO51GWud
WARNING  排除该文件，类型:bitable,名称:xx骨干核心表模板及范例
INFO     检查文件夹成功，共查到36张多维表格参与联查，如下：
INFO     医药
INFO     钠离子电池
INFO     通信
INFO     半导体
INFO     锂离子电池
INFO     工业制造
INFO     钢铁
INFO     光伏
INFO     煤炭
INFO     电力储能
INFO     风电
INFO     贵金属
INFO     海运
INFO     消费电子
INFO     种植
INFO     汽车制造
INFO     教育
INFO     零售餐饮
INFO     广告传媒
INFO     石油天然气
INFO     银行
INFO     石油化工
INFO     互联网云计算
INFO     家居
INFO     环保
INFO     矿产
INFO     电影
INFO     食品饮料
INFO     陆运
INFO     地产
INFO     区域图
INFO     纺织业
INFO     航空航天
INFO     畜牧业
INFO     美妆
INFO     旅游


{'医药': 'I6SFbcZLnaNP8nscvFCcKDRknqf',
 '钠离子电池': 'bascn57r5wVM7sM6aApFQtS7eYb',
 '通信': 'bascnfhDdoFtGSgqD9AuNzhswLf',
 '半导体': 'bascnbzucap8DFko3LgSLe0q1Kd',
 '锂离子电池': 'ZG1YbzKjQaPjIpsmk9NcSTBZnag',
 '工业制造': 'bascnQfxCAOxlZRfN0pv1YA28Hf',
 '钢铁': 'IEWKbVaDLayP0Bskj6ucfJB8nGc',
 '光伏': 'bascngo31Pi722JP5rBWEMgQL4e',
 '煤炭': 'bascnTB3mCMQdzSb3kJ37ATgQyg',
 '电力储能': 'bascnsACceveT2H48fktBz3xK7f',
 '风电': 'bascn5vPLSXteiWCaBOBYxoN8Vh',
 '贵金属': 'bascnk6XVDWXO7mUipEnqaNnAJf',
 '海运': 'BfmAbCOziaY61NsAFk6cMa7Ynvc',
 '消费电子': 'HMN8bAg6saMwZdsrHpHcRtBfnAg',
 '种植': 'Rr4XbrLEzaargtsgfF1c9ppxnsc',
 '汽车制造': 'bascnIRb34I8uajC8l70BZYtrYe',
 '教育': 'A164bE8RCabsnCss1tlcacuPn5b',
 '零售餐饮': 'DBnAb4TSCa29aXsqWhPcD2Pmn0f',
 '广告传媒': 'VUPJbq9T3azkGLseojQcbIGNnih',
 '石油天然气': 'JGUEbKc6WaLUF3sj10ZcgRhdn7g',
 '银行': 'NQX9b6UZ6astfrsthMac8fcgnTb',
 '石油化工': 'YwFYbfJs8asYqIs2IdscNjUHn0c',
 '互联网云计算': 'bascnX3JNXBB9J428M3qHmJv1Me',
 '家居': 'DH5wbDbqAaFpz9sZc51cyJimnzg',
 '环保': 'VZcSblwVaa1lgHsZADOcEyAgnEe',
 '矿产': 'MrRSbYezbakf3

In [15]:
files

{'医药': 'I6SFbcZLnaNP8nscvFCcKDRknqf',
 '钠离子电池': 'bascn57r5wVM7sM6aApFQtS7eYb',
 '通信': 'bascnfhDdoFtGSgqD9AuNzhswLf',
 '半导体': 'bascnbzucap8DFko3LgSLe0q1Kd',
 '锂离子电池': 'ZG1YbzKjQaPjIpsmk9NcSTBZnag',
 '工业制造': 'bascnQfxCAOxlZRfN0pv1YA28Hf',
 '钢铁': 'IEWKbVaDLayP0Bskj6ucfJB8nGc',
 '光伏': 'bascngo31Pi722JP5rBWEMgQL4e',
 '煤炭': 'bascnTB3mCMQdzSb3kJ37ATgQyg',
 '电力储能': 'bascnsACceveT2H48fktBz3xK7f',
 '风电': 'bascn5vPLSXteiWCaBOBYxoN8Vh',
 '贵金属': 'bascnk6XVDWXO7mUipEnqaNnAJf',
 '海运': 'BfmAbCOziaY61NsAFk6cMa7Ynvc',
 '消费电子': 'HMN8bAg6saMwZdsrHpHcRtBfnAg',
 '种植': 'Rr4XbrLEzaargtsgfF1c9ppxnsc',
 '汽车制造': 'bascnIRb34I8uajC8l70BZYtrYe',
 '教育': 'A164bE8RCabsnCss1tlcacuPn5b',
 '零售餐饮': 'DBnAb4TSCa29aXsqWhPcD2Pmn0f',
 '广告传媒': 'VUPJbq9T3azkGLseojQcbIGNnih',
 '石油天然气': 'JGUEbKc6WaLUF3sj10ZcgRhdn7g',
 '银行': 'NQX9b6UZ6astfrsthMac8fcgnTb',
 '石油化工': 'YwFYbfJs8asYqIs2IdscNjUHn0c',
 '互联网云计算': 'bascnX3JNXBB9J428M3qHmJv1Me',
 '家居': 'DH5wbDbqAaFpz9sZc51cyJimnzg',
 '环保': 'VZcSblwVaa1lgHsZADOcEyAgnEe',
 '矿产': 'MrRSbYezbakf3

In [16]:
report_df=pd.DataFrame(columns=['level','body','backbone','rough'])
report_df

,level,body,backbone,rough


In [17]:

internal_result,report_df=work1(files,report_df)
internal_result


INFO     start


good table


CRITICAL 对 “医药”  多维表格进行内部检查
CRITICAL 本骨干标记：
CRITICAL 医药
CRITICAL 完成对 “医药”  多维表格进行内部检查



good table


CRITICAL 对 “钠离子电池”  多维表格进行内部检查
ERROR    检查到骨干定义表问题：没有勾选本骨干
CRITICAL 无法对 “钠离子电池”  多维表格进行内部检查



...

good table


CRITICAL 对 “美妆”  多维表格进行内部检查
CRITICAL 本骨干标记：
CRITICAL 美妆
CRITICAL 完成对 “美妆”  多维表格进行内部检查



good table


CRITICAL 对 “旅游”  多维表格进行内部检查
CRITICAL 本骨干标记：
CRITICAL 旅游
ERROR    出现边没有找到对应的起/终点，可能是因为起终点没有勾选粗骨干权限。id:"reczR4jhze",起点："游轮服务" 终点:"景区观光"
ERROR    出现边没有找到对应的起/终点，可能是因为起终点没有勾选粗骨干权限。id:"recb6GCgjC",起点："云顶香港" 终点:"游轮服务"
ERROR    出现边没有找到对应的起/终点，可能是因为起终点没有勾选粗骨干权限。id:"recnEMwED7",起点："休闲船艇" 终点:"游轮服务"
INFO     检查到与“地产”骨干间的正确星图边界：[商业房地产]-(原材料)->[住宿服务]
INFO     检查到与“地产”骨干间的正确星图边界：[华侨城A]-(生产)->[商业房地产]
INFO     检查到与“地产”骨干间的正确星图边界：[三湘印象]-(生产)->[商业房地产]
INFO     检查到与“地产”骨干间的正确星图边界：[新华联]-(生产)->[商业房地产]
INFO     检查到与“地产”骨干间的正确星图边界：[ 海南高速]-(生产)->[商业房地产]
INFO     检查到与“石油天然气”骨干间的正确星图边界：[航空汽油]-(原材料)->[航空运输]
WARNING  检查到本骨干之外的,属于“石油天然气”骨干的边，该边不参与冲突检查：[成品油]-(父节点)->[航空汽油]
ERROR    “客车”节点没有找到对应的外部骨干：“汽车”
ERROR    因“客车”节点没有找到对应外部骨干，与其相连的边也出错,不参与联调：[客车]-(原材料)->[景区观光]
INFO     检查到与“互联网云计算”骨干间的正确星图边界：[中科云网]-(生产)->[游戏]
INFO     检查到与“电影”骨干间的正确星图边界：[中视传媒]-(生产)->[影视制片]
CRITICAL 完成对 “旅游”  多维表格进行内部检查

CRITICAL 内部检查完成，以下表格出现问题不参与联调:['钠离子电池', '区域图']



{'I6SFbcZLnaNP8nscvFCcKDRknqf': {'internal_links': [{'bitapp_id': 'recuBdc461',
    'source': 'rec4F9dJLT',
    'source_name_ref': '原料药',
    'target_name_ref': '化学药',
    'target': 'rec4V0VCNn',
    'category': '原材料',
    'more_info': {},
    'target_name': '化学药',
    'source_name': '原料药'},
   {'bitapp_id': 'recylGEN8f',
    'source': 'rec6yDTGFq',
    'source_name_ref': '医药中间体',
    'target_name_ref': '原料药',
    'target': 'rec4F9dJLT',
    'category': '原材料',
    'more_info': {},
    'target_name': '原料药',
    'source_name': '医药中间体'},
   {'bitapp_id': 'recTXmCnXy',
    'source': 'recGqKtjof',
    'source_name_ref': '化合物研究服务',
    'target_name_ref': '药品配方',
    'target': 'recvrZmGcK',
    'category': '原材料',
    'more_info': {},
    'source_name': '化合物研究服务',
    'target_name': '药品配方'},
   {'bitapp_id': 'receAfFSC9',
    'source': 'rec3idm4QX',
    'source_name_ref': '试验用动物',
    'target_name_ref': '化合物研究服务',
    'target': 'recGqKtjof',
    'category': '原材料',
    'more_info': {},
    'tar

In [18]:
result,report_df=work2(internal_result,files,report_df)



CRITICAL 开始外部联调，参与表格为：
CRITICAL ['医药', '钠离子电池', '通信', '半导体', '锂离子电池', '工业制造', '钢铁', '光伏', '煤炭', '电力储能', '风电', '贵金属', '海运', '消费电子', '种植', '汽车制造', '教育', '零售餐饮', '广告传媒', '石油天然气', '银行', '石油化工', '互联网云计算', '家居', '环保', '矿产', '电影', '食品饮料', '陆运', '地产', '区域图', '纺织业', '航空航天', '畜牧业', '美妆', '旅游']

CRITICAL 对标记为"医药"的表格的边界进行检查,该表共有0条外边参与联调
CRITICAL 医药
CRITICAL 完成对标记为"医药"的表格的边界进行检查,该表共有0条外边参与联调，通过边0条，冲突边0条

CRITICAL 对标记为"通信"的表格的边界进行检查,该表共有3条外边参与联调
CRITICAL 通信
ERROR    星图边界识别失败，该边："互联网数据中心--原材料->宽带通信服务"在对应"互联网云计算"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："手机--原材料->移动通信需求"在对应"消费电子"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："天线模块--原材料->手机"在对应"消费电子"骨干中没找到对应边界
CRITICAL 完成对标记为"通信"的表格的边界进行检查,该表共有3条外边参与联调，通过边0条，冲突边3条

CRITICAL 对标记为"半导体"的表格的边界进行检查,该表共有73条外边参与联调
CRITICAL 半导体
ERROR    星图边界识别失败，该边："创益通--生产->电连接器"在对应"航空航天"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："电连技术--生产->电连接器"在对应"航空航天"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："中航光电--生产->电连接器"在对应"航空航天"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："航天电器--生产->电连接器"在对应"航空航天"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："维峰电子--生产->电连

{'bitapp_id': 'rectDUbCoD', 'source': 'recOSjwToq', 'source_name_ref': '功率模块', 'target_name_ref': '储能逆变器', 'category': '原材料', 'more_info': {}, 'target_name': '储能逆变器', 'source_name': '功率模块', 'external_backbone_name': '电力储能', 'external_backbone': 'bascnsACceveT2H48fktBz3xK7f'}
{'bitapp_id': 'recDNWOrhe', 'source_name_ref': '功率模块', 'target_name_ref': '储能逆变器', 'target': 'recNYsT9kW', 'category': '原材料', 'more_info': {}, 'target_name': '储能逆变器', 'source_name': '功率模块', 'external_backbone_name': '半导体', 'external_backbone': 'bascnbzucap8DFko3LgSLe0q1Kd'}
{'bitapp_id': 'rec7KW9GWC', 'source': 'rec4i53UgC', 'source_name_ref': 'ROM', 'target_name_ref': '存储服务器', 'category': '原材料', 'more_info': {}, 'source_name': 'ROM', 'target_name': '存储服务器', 'external_backbone_name': '互联网云计算', 'external_backbone': 'bascnX3JNXBB9J428M3qHmJv1Me'}
{'bitapp_id': 'reczzGzIkM', 'source_name_ref': 'ROM', 'target_name_ref': '存储服务器', 'target': 'recqS3IYxB', 'category': '原材料', 'more_info': {}, 'target_name': '存储服务器', 'source

ERROR    星图边界识别失败，该边："电池后段设备--原材料->锂电池模组"在对应"工业制造"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："电池后段设备--原材料->锂离子电池"在对应"工业制造"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："杭可科技--生产->电池后段设备"在对应"工业制造"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："华自科技--生产->电池后段设备"在对应"工业制造"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："星云股份--生产->电池后段设备"在对应"工业制造"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："电池前段设备--原材料->锂电池正极极片"在对应"工业制造"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："电池前段设备--原材料->锂电池负极极片"在对应"工业制造"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："电池前段设备--父节点->涂布机"在对应"工业制造"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："电池前段设备--父节点->真空搅拌机"在对应"工业制造"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："电池前段设备--父节点->辊压机"在对应"工业制造"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："电池前段设备--父节点->分切机"在对应"工业制造"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："电池前段设备--父节点->制片机"在对应"工业制造"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："海目星--生产->电池前段设备"在对应"工业制造"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："金银河--生产->电池前段设备"在对应"工业制造"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："电池中段设备--父节点->卷绕机"在对应"工业制造"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："电池中段设备--父节点->叠片机"在对应"工业制造"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："电池中段设备--父节点->电芯入壳机"在对应

{'bitapp_id': 'recOghMvMS', 'source': 'recE3vQsGy', 'source_name_ref': '动力锂电池', 'target_name_ref': '电源系统', 'category': '原材料', 'more_info': {}, 'target_name': '电源系统', 'source_name': '动力锂电池', 'external_backbone_name': '汽车制造', 'external_backbone': 'bascnIRb34I8uajC8l70BZYtrYe'}
{'bitapp_id': 'recJPVmIlE', 'source_name_ref': '动力锂电池', 'target_name_ref': '电源系统', 'target': 'recl9QXaoE', 'category': '原材料', 'more_info': {}, 'target_name': '电源系统', 'source_name': '动力锂电池', 'external_backbone_name': '锂离子电池', 'external_backbone': 'ZG1YbzKjQaPjIpsmk9NcSTBZnag'}
{'bitapp_id': 'recoDDOreA', 'source': 'recSPgKJKd', 'source_name_ref': '钢型材', 'target_name_ref': '钢轨', 'category': '父节点', 'more_info': {}, 'source_name': '钢型材', 'target_name': '钢轨', 'external_backbone_name': '陆运', 'external_backbone': 'bascnipf2OKC7NYIMkWvAe5Fwhf'}
{'bitapp_id': 'recTXmCnXy', 'source_name_ref': '钢型材', 'target_name_ref': '钢轨', 'target': 'recE3vQsGy', 'category': '父节点', 'more_info': {}, 'target_name': '钢轨', 'source_name': '钢型材',

INFO     星图边界识别成功，在"陆运"骨干中查到对应边："钢带--原材料->铁路桥梁"
ERROR    星图边界识别失败，该边："轧钢板--原材料->汽车骨架"在对应"汽车制造"骨干中没找到对应边界
INFO     星图边界识别成功，在"海运"骨干中查到对应边："轧钢板--原材料->集装箱"
ERROR    星图边界识别失败，该边："余热锅炉--原材料->电力"在对应"电力储能"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："钨铁--原材料->炼钢"在对应"矿产"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："不锈钢--原材料->医疗器械"在对应"医药"骨干中没找到对应边界
INFO     星图边界识别成功，在"海运"骨干中查到对应边："干散货航运--储运->铁矿石"
ERROR    星图边界识别失败，该边："港口--储运->铁矿石"在对应"海运"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："型钢--原材料->船舶"在对应"海运"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："铝合金--原材料->船舶"在对应"海运"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："铸件，锻件--父节点->医疗器械零部件"在对应"医药"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："铝合金--原材料->高压导线"在对应"电力储能"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："铝铜合金--原材料->航天器零部件"在对应"航空航天"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："白铜--原材料->精密机械"在对应"工业制造"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："青铜--原材料->轴承"在对应"工业制造"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："铸件，锻件--父节点->航空发动机零部件"在对应"航空航天"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："钛合金--原材料->航空发动机零部件"在对应"航空航天"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："铝合金--原材料->航空器外壳，航空器骨架"在对应"航空航天"骨干中没找到对应边界
INF

...

{'bitapp_id': 'recsodarpl', 'source': 'reckYl1q4C', 'source_name_ref': '嘉华股份', 'target_name_ref': '大豆油', 'category': '生产', 'more_info': {}, 'target_name': '大豆油', 'source_name': '嘉华股份', 'external_backbone_name': '种植', 'external_backbone': 'Rr4XbrLEzaargtsgfF1c9ppxnsc'}
{'bitapp_id': 'reciU8qqFN', 'source': 'reciUbnYLh', 'source_name_ref': '嘉华股份', 'target_name_ref': '大豆油', 'category': '生产', 'more_info': {}, 'target_name': '大豆油', 'source_name': '嘉华股份', 'external_backbone_name': '食品饮料', 'external_backbone': 'BQfjb41pka6j8os5iIZcqCpjnV1'}
{'bitapp_id': 'recrBWGvOg', 'source': 'recJKHzh9I', 'source_name_ref': '商品蛋', 'target_name_ref': '鲜鸡蛋', 'category': '父节点', 'more_info': {}, 'source_name': '商品蛋', 'target_name': '鲜鸡蛋', 'external_backbone_name': '畜牧业', 'external_backbone': 'HmoFbzbXNakHFpsyCDUctlOTnIb'}
{'bitapp_id': 'recgLBNlns', 'source_name_ref': '商品蛋', 'target_name_ref': '鲜鸡蛋', 'target': 'recpcDStt8', 'category': '父节点', 'more_info': {}, 'target_name': '鲜鸡蛋', 'source_name': '商品蛋', 'extern

ERROR    星图边界识别失败，该边："家用电器--原材料->住宅装修"在对应"家居"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："美的集团--生产->家用电器"在对应"家居"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："格力电器--生产->家用电器"在对应"家居"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："四川长虹--生产->家用电器"在对应"家居"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："永安林业--生产->木材"在对应"种植"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："C森泰股份--生产->木材"在对应"种植"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："起重机钢轨--原材料->起重机"在对应"钢铁"骨干中没找到对应边界
INFO     星图边界识别成功，在"电影"骨干中查到对应边："商业房地产--原材料->院线放映"
ERROR    星图边界识别失败，该边："华联股份--生产->院线放映"在对应"电影"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："嘉凯城--生产->院线放映"在对应"电影"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："好太太--生产->智能家居"在对应"家居"骨干中没找到对应边界
INFO     星图边界识别成功，在"旅游"骨干中查到对应边："商业房地产--原材料->住宿服务"
CRITICAL 完成对标记为"地产"的表格的边界进行检查,该表共有58条外边参与联调，通过边11条，冲突边47条

CRITICAL 对标记为"纺织业"的表格的边界进行检查,该表共有49条外边参与联调
CRITICAL 纺织业
ERROR    星图边界识别失败，该边："面料--父节点->化纤面料"在对应"石油化工"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："常山北明--生产->化纤面料"在对应"石油化工"骨干中没找到对应边界
INFO     星图边界识别成功，在"种植"骨干中查到对应边："天然纤维--父节点->棉"
INFO     星图边界识别成功，在"种植"骨干中查到对应边："棉--原材料->再生纤维素纤维"
ERROR    星图边界识别失败，该边："联发股份

{'bitapp_id': 'rec7boTbMB', 'source': 'rec3C33UU3', 'source_name_ref': '商业房地产', 'target_name_ref': '院线放映', 'category': '原材料', 'more_info': {}, 'source_name': '商业房地产', 'target_name': '院线放映', 'external_backbone_name': '电影', 'external_backbone': 'bascnCFrkVGczkkyWTXqAdLTLqe'}
{'bitapp_id': 'recNyxhy9w', 'source_name_ref': '商业房地产', 'target_name_ref': '院线放映', 'target': 'recRamnp5z', 'category': '原材料', 'more_info': {}, 'target_name': '院线放映', 'source_name': '商业房地产', 'external_backbone_name': '地产', 'external_backbone': 'HRwdbhrmTaAOBws3hLhcKhKbnlh'}
{'bitapp_id': 'recaFvjhdY', 'source': 'rec3C33UU3', 'source_name_ref': '商业房地产', 'target_name_ref': '住宿服务', 'category': '原材料', 'more_info': {}, 'source_name': '商业房地产', 'target_name': '住宿服务', 'external_backbone_name': '旅游', 'external_backbone': 'bascnoh0v8aKAFhMqihl6dxxTdf'}
{'bitapp_id': 'recscetuwn', 'source_name_ref': '商业房地产', 'target_name_ref': '住宿服务', 'target': 'recoBIz2VC', 'category': '原材料', 'more_info': {'pro_line': '主营'}, 'target_name': '住宿服

ERROR    星图边界识别失败，该边："亚光科技--生产->射频器件"在对应"半导体"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："中航电测--生产->飞机"在对应"电力储能"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："合成树脂--原材料->预浸料"在对应"石油化工"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："聚丙烯腈--原材料->碳纤维预浸料"在对应"石油化工"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："聚氯乙烯--原材料->隔热材料"在对应"石油化工"骨干中没找到对应边界
INFO     星图边界识别成功，在"半导体"骨干中查到对应边："逻辑芯片--原材料->航空电子"
ERROR    星图边界识别失败，该边："铝合金--原材料->火箭发动机外壳"在对应"钢铁"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："宝钛股份--生产->钛合金"在对应"钢铁"骨干中没找到对应边界
INFO     星图边界识别成功，在"钢铁"骨干中查到对应边："钛合金--原材料->航空器外壳，航空器骨架"
CRITICAL 完成对标记为"航空航天"的表格的边界进行检查,该表共有26条外边参与联调，通过边2条，冲突边24条

CRITICAL 对标记为"畜牧业"的表格的边界进行检查,该表共有24条外边参与联调
CRITICAL 畜牧业
INFO     星图边界识别成功，在"种植"骨干中查到对应边："玉米--原材料->能量原材料"
INFO     星图边界识别成功，在"种植"骨干中查到对应边："小麦--原材料->能量原材料"
INFO     星图边界识别成功，在"种植"骨干中查到对应边："高粱--原材料->能量原材料"
INFO     星图边界识别成功，在"种植"骨干中查到对应边："豆粕--原材料->植物性蛋白"
INFO     星图边界识别成功，在"种植"骨干中查到对应边："菜粕--原材料->植物性蛋白"
INFO     星图边界识别成功，在"食品饮料"骨干中查到对应边："牛奶--原材料->乳制品"
ERROR    星图边界识别失败，该边："伊利股份--生产->乳制品"在对应"食品饮料"骨干中没找到对应边界
ERROR    星图边界识别失败，该边："肉鸭--原材料

{'bitapp_id': 'recmOTfpwV', 'source': 'recvVidZzf', 'source_name_ref': '牛奶', 'target_name_ref': '乳制品', 'category': '原材料', 'more_info': {}, 'source_name': '牛奶', 'target_name': '乳制品', 'external_backbone_name': '食品饮料', 'external_backbone': 'BQfjb41pka6j8os5iIZcqCpjnV1'}
{'bitapp_id': 'recpjqBpj3', 'source_name_ref': '牛奶', 'target_name_ref': '乳制品', 'target': 'rec8OHrc2n', 'category': '原材料', 'more_info': {}, 'source_name': '牛奶', 'target_name': '乳制品', 'external_backbone_name': '畜牧业', 'external_backbone': 'HmoFbzbXNakHFpsyCDUctlOTnIb'}
{'bitapp_id': 'recKurD4j9', 'source': 'recpcDStt8', 'source_name_ref': '鲜鸡蛋', 'target_name_ref': '烘焙食品', 'category': '原材料', 'more_info': {}, 'source_name': '鲜鸡蛋', 'target_name': '烘焙食品', 'external_backbone_name': '食品饮料', 'external_backbone': 'BQfjb41pka6j8os5iIZcqCpjnV1'}
{'bitapp_id': 'rec4q4VAzE', 'source_name_ref': '鲜鸡蛋', 'target_name_ref': '烘焙食品', 'target': 'recsouIpQc', 'category': '原材料', 'more_info': {}, 'source_name': '鲜鸡蛋', 'target_name': '烘焙食品', 'extern

In [48]:
report_df

,level,body,backbone,rough
0,error,"出现边没有找到对应的起/终点，可能是因为起终点没有勾选粗骨干权限。id:""rec3f1Oy9...",零售餐饮,True
0,warning,"检查到本骨干之外的,属于“石油天然气”骨干的边，该边不参与冲突检查：[英国布伦特原油]-(反...",海运,True
0,warning,"检查到本骨干之外的,属于“银行”骨干的边，该边不参与冲突检查：[房地产开发贷款]-(父节点)...",地产,True
0,warning,"检查到本骨干之外的,属于“银行”骨干的边，该边不参与冲突检查：[房地产开发贷款]-(父节点)...",地产,True
0,warning,"检查到本骨干之外的,属于“银行”骨干的边，该边不参与冲突检查：[房地产开发贷款余额]-(反映...",地产,True
...,...,...,...,...
0,error,"星图边界识别失败，该边：""丁苯橡胶--原材料->电线电缆""在对应""电力储能""骨干中没找到对应边界",石油化工,True
0,error,"星图边界识别失败，该边：""POE粒子--原材料->POE胶膜""在对应""光伏""骨干中没找到对应边界",石油化工,True
0,error,"星图边界识别失败，该边：""煤制乙二醇工艺--原材料->乙二醇""在对应""煤炭""骨干中没找到对应边界",石油化工,True
0,error,"星图边界识别失败，该边：""合成气--原材料->煤制乙二醇工艺""在对应""煤炭""骨干中没找到对应边界",石油化工,True


In [19]:
from datetime import datetime
d=datetime.today().__format__("%Y%m%d")
report_df.to_csv(r'E:\wangzhilin\QuantumGalaxy\opt\combine_check\report-%s.csv'%d)

In [20]:
dup=find_common_elements(result,files)
dup=[[d[0],d[1][0],d[1][1]] for d in dup]
dup
import pandas as pd
dupdf=pd.DataFrame(data=dup,columns=['节点名','所属骨干1','所属骨干2'])
dupdf.sort_values('节点名')

dupdf.to_csv(r'E:\wangzhilin\QuantumGalaxy\opt\combine_check\dupnames.csv')
dupdf

,节点名,所属骨干1,所属骨干2
0,氩气,医药,钢铁
1,淀粉,医药,种植
2,玉米,医药,种植
3,天然气,医药,石油天然气
4,丙烯,医药,石油化工
...,...,...,...
201,肉鸭,食品饮料,畜牧业
202,饮食类需求,食品饮料,畜牧业
203,牛肉,食品饮料,畜牧业
204,肉牛,食品饮料,畜牧业


In [21]:
def save_json(result):
    import json
    
    d=datetime.today().__format__("%Y%m%d")
    b = json.dumps(result)
    f2 = open(r'E:\wangzhilin\QuantumGalaxy\opt\combine_check\results\result-%s.json'%d, 'w')
    f2.write(b)
    f2.close()
save_json(result)

# result2cypher

对所有表格里的internalnodes internallinks+ good_external_links就ok？（goodexternallink 有两组,可能moreinfo还不相同,用merge？不相同时会有重边。需要提前融合外边

重名内部节点处理？ 公司名字有唯一性约束，其他没有。merge会对{}内的项进行完全匹配，内部重名和连线不会出错（需要联调时检查并warning）。只需要把公司做更新backbone和code
连边的时候需要对点明确骨干？bitappid！

node的数据项 backbone，配图（地址，后续做,backbonetoken+bitappid，后续每次刷库时可以用md5验证图片之后再下到本地？api没有md5，必须要下载吧），code,category,（x,y）

link的数据项 sourcename type targetname (产线地位)， 只用名字匹配？？

其他信息及其结构要改82的bitablework和81的飞书接口

In [22]:

vertexs=[]
links={}


nodedic={'产品':'Product','企业':'Company','指标':'Indicator','工艺':'Technique','需求':'Demand'}
linkdic={'原材料':'compose','供应':'supply','父节点':'parent','储运':'carry','生产':'produce','同位':'peer','反映':'measure'}
for file_token,data in result.items():
    print(data['backbone_mark'])
    print('内部节点:%s'%len(data['internal_nodes']))
    print('内部边:%s'%len(data['internal_links']))
    print('外部节点:%s'%len(data['external_nodes']))
    print('外部边:%s'%len(data['good_external_links']))
    for node in data['internal_nodes']:
        post_node=deepcopy(node)
        cate=post_node.pop('category')
        cate_new=nodedic.get(cate,cate)
        post_node['category']=cate_new
        more=post_node.pop('more_info')
        post_node['file_token']=file_token
        post_node['backbone']=data['backbone_mark']
        if more:
            post_node.update(more)
        vertexs.append(post_node)
    for link in data['internal_links']:
        post_link={}
        post_link['source']=link['source']
        post_link['target']=link['target']
        post_link['target_name']=link['target_name']
        post_link['source_name']=link['source_name']
        post_link['category']=linkdic.get(link['category'],link['category'])
        if link.get('more_info'):
            
            post_link.update(link['more_info'])
        key=link['source_name']+'--'+link['category']+'->'+link['target_name']
        if key not in links.keys():
            links[key]=post_link
        else:
            body="“%s”骨干内部重复的边:'%s'"%(data['backbone_mark'],key)
            logger.warning(body)
            report_df=append_df(report_df,['warning',body,data['backbone_mark'],True])
            links[key].update(post_link)
    for link in data['good_external_links']:
        #要找到其节点所在本骨干的recordid，不能直接用link里的id。。
        post_link={}
        post_link['source']=link['source']
        post_link['target']=link['target']
        post_link['target_name']=link['target_name']
        post_link['source_name']=link['source_name']
        post_link['category']=linkdic.get(link['category'],link['category'])
        if link.get('more_info'):
            post_link.update(link['more_info'])
        key=link['source_name']+'--'+link['category']+'->'+link['target_name']
        if key not in links.keys():
            links[key]=post_link
        else:
            #print("dup!:'%s'"%key)
            links[key].update(post_link)
post_result={'vertex':vertexs,'link':list(links.values())}
print(len(post_result['vertex']))
print(len(post_result['link']))
        


WARNING  “半导体”骨干内部重复的边:'华兴源创--生产->芯片封装测试设备'
WARNING  “半导体”骨干内部重复的边:'朗科科技--生产->内存'
WARNING  “半导体”骨干内部重复的边:'北方华创--生产->半导体薄膜沉积设备'
WARNING  “半导体”骨干内部重复的边:'北方华创--生产->刻蚀机'
WARNING  “半导体”骨干内部重复的边:'北方华创--生产->半导体清洗设备'
WARNING  “半导体”骨干内部重复的边:'澜起科技--生产->电源管理芯片，电源管理IC'
WARNING  “半导体”骨干内部重复的边:'澜起科技--生产->存储芯片'
WARNING  “半导体”骨干内部重复的边:'润欣科技--生产->电子元器件分销服务'
WARNING  “半导体”骨干内部重复的边:'力源信息--生产->电子元器件分销服务'
WARNING  “半导体”骨干内部重复的边:'雅创电子--生产->电子元器件分销服务'
WARNING  “锂离子电池”骨干内部重复的边:'赣锋锂业--生产->电池级碳酸锂'
WARNING  “锂离子电池”骨干内部重复的边:'赣锋锂业--生产->氢氧化锂'
WARNING  “锂离子电池”骨干内部重复的边:'天齐锂业--生产->电池级碳酸锂'
WARNING  “锂离子电池”骨干内部重复的边:'天齐锂业--生产->氢氧化锂'
WARNING  “工业制造”骨干内部重复的边:'控制类设备--原材料->通用自动化控制系统'
WARNING  “工业制造”骨干内部重复的边:'传动类装置--原材料->通用自动化控制系统'
WARNING  “工业制造”骨干内部重复的边:'驱动类设备--原材料->通用自动化控制系统'
WARNING  “工业制造”骨干内部重复的边:'传感类装置--原材料->通用自动化控制系统'
WARNING  “工业制造”骨干内部重复的边:'伺服电机--原材料->伺服系统'
WARNING  “工业制造”骨干内部重复的边:'磁材--原材料->伺服电机'
WARNING  “工业制造”骨干内部重复的边:'滚珠丝杠--原材料->执行器'
WARNING  “工业制造”骨干内部重复的边:'步进电机--原材料->步进系统'
WARNING  “工业制造”骨干内部重复的边:'编码器--原材料->伺服系

医药
内部节点:328
内部边:270
外部节点:0
外部边:0
通信
内部节点:205
内部边:101
外部节点:4
外部边:0
半导体
内部节点:488
内部边:586
外部节点:37
外部边:7
锂离子电池
内部节点:306
内部边:426
外部节点:18
外部边:2
工业制造
内部节点:566
内部边:636
外部节点:2
外部边:0


WARNING  “工业制造”骨干内部重复的边:'行业专用设备--父节点->半导体设备'
WARNING  “工业制造”骨干内部重复的边:'行业专用设备--父节点->工程机械设备'
WARNING  “工业制造”骨干内部重复的边:'行业专用设备--父节点->光伏设备'
WARNING  “工业制造”骨干内部重复的边:'行业专用设备--父节点->高端仪器'
WARNING  “工业制造”骨干内部重复的边:'行业专用设备--父节点->3C产品生产设备'
WARNING  “工业制造”骨干内部重复的边:'行业专用设备--父节点->汽车生产设备'
WARNING  “工业制造”骨干内部重复的边:'行业专用设备--父节点->其他行业生产设备'
WARNING  “工业制造”骨干内部重复的边:'行业专用设备--父节点->石化行业设备'
WARNING  “工业制造”骨干内部重复的边:'执行层--父节点->机器人'
WARNING  “工业制造”骨干内部重复的边:'执行层--父节点->数控机床'
WARNING  “工业制造”骨干内部重复的边:'机床主体--原材料->模具'
WARNING  “工业制造”骨干内部重复的边:'行业专用设备--父节点->空气分离设备'
WARNING  “工业制造”骨干内部重复的边:'空气分离设备--原材料->工业气体'
WARNING  “工业制造”骨干内部重复的边:'液压密封产品--原材料->液压系统'
WARNING  “工业制造”骨干内部重复的边:'其他工业原材料--原材料->通用部件与服务'
WARNING  “工业制造”骨干内部重复的边:'其他行业生产设备--父节点->核工业设备'
WARNING  “工业制造”骨干内部重复的边:'工程设计与咨询--原材料->工程承包'
WARNING  “工业制造”骨干内部重复的边:'工业网络设备--原材料->智能工厂'
WARNING  “工业制造”骨干内部重复的边:'感知设备--原材料->智能工厂'
WARNING  “工业制造”骨干内部重复的边:'控制类设备--原材料->通用自动化控制系统'
WARNING  “工业制造”骨干内部重复的边:'传动类装置--原材料->通用自动化控制系统'
WARNING  “工业制造”骨干内部重复的边:'驱动类设备--原材料->通用自动化控制系统'
WARNING 

钢铁
内部节点:136
内部边:141
外部节点:45
外部边:9
光伏
内部节点:232
内部边:276
外部节点:6
外部边:2
煤炭
内部节点:106
内部边:108
外部节点:5
外部边:2
电力储能
内部节点:498
内部边:578
外部节点:15
外部边:5
风电
内部节点:135
内部边:156
外部节点:2
外部边:1


WARNING  “风电”骨干内部重复的边:'龙源电力--生产->风力发电'
WARNING  “风电”骨干内部重复的边:'金风科技--生产->风电整机'
WARNING  “风电”骨干内部重复的边:'金风科技--生产->风电发电机组'
WARNING  “风电”骨干内部重复的边:'金风科技--生产->风电场'
WARNING  “贵金属”骨干内部重复的边:'豫光金铅--生产->标准银'
WARNING  “海运”骨干内部重复的边:'青岛港--生产->码头运营'
WARNING  “海运”骨干内部重复的边:'秦港股份--生产->码头运营'
WARNING  “海运”骨干内部重复的边:'中船防务--生产->舰船维修'
WARNING  “海运”骨干内部重复的边:'中国外运--生产->物流服务'
WARNING  “海运”骨干内部重复的边:'中远海控--生产->集装箱航运'
WARNING  “海运”骨干内部重复的边:'中远海能--生产->原油航运'
WARNING  “汽车制造”骨干内部重复的边:'双环传动--生产->传动系统'
WARNING  “汽车制造”骨干内部重复的边:'征和工业--生产->传动系统'
WARNING  “汽车制造”骨干内部重复的边:'科达利--生产->锂电池结构件'
WARNING  “汽车制造”骨干内部重复的边:'新宙邦--生产->锂电池电解液'
WARNING  “汽车制造”骨干内部重复的边:'轮胎--原材料->车轮'
WARNING  “汽车制造”骨干内部重复的边:'汽车内饰--原材料->汽车车身'
WARNING  “汽车制造”骨干内部重复的边:'上汽集团--生产->汽车'
WARNING  “人力资源（教育）”骨干内部重复的边:'新华文轩--生产->图书期刊出版'
WARNING  “人力资源（教育）”骨干内部重复的边:'新华文轩--生产->图书期刊发行'
WARNING  “广告传媒”骨干内部重复的边:'芒果超媒--生产->广播电视媒体'
WARNING  “广告传媒”骨干内部重复的边:'城市传媒--生产->出版物'
WARNING  “广告传媒”骨干内部重复的边:'分众传媒--生产->场景媒体'
WARNING  “广告传媒”骨干内部重复的边:'易点天下--生产->广告营销'
WARNING  “石油化工”骨干内部重复的边:'苯

...

电影
内部节点:107
内部边:141
外部节点:3
外部边:1
食品饮料
内部节点:434
内部边:361
外部节点:50
外部边:9
陆运
内部节点:102
内部边:104
外部节点:8
外部边:2
地产
内部节点:521
内部边:602
外部节点:25
外部边:11
纺织业
内部节点:110
内部边:89
外部节点:54
外部边:24
航空航天
内部节点:199
内部边:204
外部节点:21
外部边:2
畜牧业
内部节点:185
内部边:205
外部节点:27
外部边:11


WARNING  “美妆”骨干内部重复的边:'美妆产品（化妆品）--父节点->彩妆类'
WARNING  “美妆”骨干内部重复的边:'美妆产品（化妆品）--父节点->护肤品类'
WARNING  “美妆”骨干内部重复的边:'护肤品类--父节点->面霜'
WARNING  “美妆”骨干内部重复的边:'护肤品类--父节点->护肤乳液'
WARNING  “美妆”骨干内部重复的边:'护肤品类--父节点->精华类产品'
WARNING  “美妆”骨干内部重复的边:'精华类产品--父节点->次抛精华'
WARNING  “美妆”骨干内部重复的边:'精华类产品--父节点->普通精华'
WARNING  “美妆”骨干内部重复的边:'活性成分--原材料->护肤品类'
WARNING  “美妆”骨干内部重复的边:'护肤品类--父节点->面膜类产品'
WARNING  “美妆”骨干内部重复的边:'面膜类产品--父节点->片状面膜'
WARNING  “美妆”骨干内部重复的边:'面膜类产品--父节点->涂抹式面膜'
WARNING  “美妆”骨干内部重复的边:'无纺布--原材料->片状面膜'
WARNING  “美妆”骨干内部重复的边:'片状面膜--父节点->冻干面膜'
WARNING  “美妆”骨干内部重复的边:'低温冻干技术--原材料->冻干面膜'
WARNING  “美妆”骨干内部重复的边:'护肤品类--父节点->清洁类产品'
WARNING  “美妆”骨干内部重复的边:'清洁类产品--父节点->卸妆产品'
WARNING  “美妆”骨干内部重复的边:'美妆产品（化妆品）--父节点->工具类'
WARNING  “美妆”骨干内部重复的边:'工具类--父节点->化妆刷'
WARNING  “美妆”骨干内部重复的边:'工具类--父节点->美妆蛋'
WARNING  “美妆”骨干内部重复的边:'乳胶--原材料->美妆蛋'
WARNING  “美妆”骨干内部重复的边:'亲水性聚氨酯（PU）--原材料->美妆蛋'
WARNING  “美妆”骨干内部重复的边:'保湿剂--原材料->护肤品类'
WARNING  “美妆”骨干内部重复的边:'工具类--父节点->化妆棉'
WARNING  “美妆”骨干内部重复的边:'棉--原材料->化妆棉'
WARNING  “美妆”骨干内部重复的边:'聚丙烯纤维

美妆
内部节点:61
内部边:102
外部节点:0
外部边:0
旅游
内部节点:189
内部边:187
外部节点:5
外部边:1


WARNING  “旅游”骨干内部重复的边:'曲江文旅--生产->景区观光'
WARNING  “旅游”骨干内部重复的边:'金陵饭店--生产->餐饮服务'
WARNING  “旅游”骨干内部重复的边:'西藏旅游--生产->景区观光'


7983
8434


In [23]:
from QGI.neoapi import Neo4j
#neo4j+ssc://534ea9b7.databases.neo4j.io:7687
#QuantumGalaxy
#atlas
uri = "neo4j+ssc://08ef0a79.databases.neo4j.io"
user = "QG_Editor"
password = "editor"
#neo1=Neo4j(uri,user,password)
#nova
uri = "neo4j+ssc://534ea9b7.databases.neo4j.io:7687"
user = "neo4j"
password = "QuantumGalaxy"

neo=Neo4j(uri,user,password)
#neo.simple_query('match (n) detach delete n')



In [24]:

def data2cypher(data):
    cyphers=['match (n) detach delete n']
    from copy import deepcopy
    d=deepcopy(data)
    for v in d['vertex']:
        #merge 应该只有名字？和骨干？公司怎么办
        cate=v.pop('category')
        if cate=='Company' or cate=='Indicator':
            name=v.pop('name')
            backbone=v.pop('backbone')
            
            cypher='merge(n:%s {name:"%s"}) \
                on match set n.backbone=n.backbone+",%s" \
                on create set n.backbone="%s" \
                '%(cate,name,backbone,backbone)
            if v:
                other='with n set '
                
                for key,value in v.items():
                    if type(value)==str:
                        value=value.replace('"', '\\\"').replace("'", "\\\'")
                    other+='n.%s="%s",'%(key,value)
                other=other[:-1]
                cypher+=other
            cyphers.append(cypher)
        else:
            s='{'
            for key,value in v.items():
                s+=key+':'+'"%s",'%value
            s=s[:-1]
            s+='}'
            cypher='merge (n:%s %s)'%(cate,s)
            #print(cypher)
            cyphers.append(cypher)

            
            #cypher='merge (n:%s {name:"%s"}) set n.backbone=n.backbone+",%s"'%(cate,v['name'],v['backbone'])
            #print('try this instead:"%s"'%cypher)
            #try:neo.simple_query(cypher)
            #except:print('failed! pass')
    for l in d['link']:
        #link表中对name和bitapp同时匹配 match (n {name:"",bitapp_id:""})
        #边界是不是不对？eg 畜牧业 牛奶- 食品饮料-乳制品
        sourcename,source,targetname,target,cate=l.pop('source_name'),l.pop('source'),l.pop('target_name'),l.pop('target'),l.pop('category')
        more=''
        if l:
            for k,v in l.items():
                more+=k+':'+'"%s",'%v
            more=more[:-1]
        if cate=='produce':
            cypher='match (n {name:"%s"}),(n1 {name:"%s",bitapp_id:"%s"}) create (n)-[r:%s {%s}]->(n1)'%(sourcename,targetname,target,cate,more) 
        else:
            cypher='match (n {name:"%s",bitapp_id:"%s"}),(n1 {name:"%s",bitapp_id:"%s"}) create (n)-[r:%s {%s}]->(n1)'%(sourcename,source,targetname,target,cate,more)          
        '''                                                                                                                                                                                   -------------------------------------+'
        s=deepcopy(data['vertex'][e['source']])
        s_cate=s.pop('category')
        s1='{'
        for key,value in s.items():
            s1+=key+':'+'"%s",'%value
        s1=s1[:-1]
        s1+='}'
        
        t=deepcopy(data['vertex'][e['target']])
        t_cate=t.pop('category')
        s2='{'
        for key,value in t.items():
            s2+=key+':'+'"%s",'%value
        s2=s2[:-1]
        s2+='}'
        cypher='match (n:%s %s),(n1:%s %s) create (n)-[r:%s]->(n1)'%(s_cate,s1,t_cate,s2,e["rela"])
        '''
        cyphers.append(cypher)
        #print(cypher)
        #neo.simple_query(cypher)
    return cyphers
cyphers=data2cypher(post_result)
print(len(cyphers))

16418


In [25]:
r=neo.multi_cypher(cyphers)
r

16418

In [ ]:
r'20	112.81 \
400	-170'


In [ ]:
data=bitable_works('bascnzsAjUXwr4gUkaIpqi8BBaf',subgraph_name='粗骨干图')
data

In [ ]:
from copy import deepcopy
#data1=deepcopy(data)

In [ ]:
import time

In [ ]:
s=time.time()
time.sleep(5)
e=time.time()-s
e

In [ ]:
fold='fldcn2VlYvt8Efb9PqZwO51GWud'

In [ ]:

r=requests.get('http://82.156.248.152:81/fs/fold_meta?token=%s'%fold).json()
r


In [ ]:
r=requests.get('http://82.156.248.152:81/fs/fold_children?fld_token=%s'%fold).json()
r

In [ ]:
r.json()

In [ ]:
import simplejson
with open(r'E:\wangzhilin\QuantumGalaxy\opt\combine_check\results\result-20230404.json','r') as file:
    data=simplejson.load(file)

In [ ]:
web=result['bascnX3JNXBB9J428M3qHmJv1Me']
web

In [ ]:
json.dumps('键桥科技')

### 多维表格附件图片的相关杂务

将所有多维表格中的配图列按照appid/tblid/recid的路径保存二进制文件（需要格式？）到8383/static/img下，将路径保存到哪里？（neo，则需要维护所有文件的node表，包括node表token

多为表格存的是图片。。neo要存路径，联调需要对所有节点存路径

In [ ]:
from html import escape
import json
from urllib.parse import quote
import requests,json

In [ ]:
bitapp_auth={'app_id':'cli_a1fc0a192db8100c','app_secret':'nNk1fT8h2eVi8B2ymIFangHTpYrOAJTD'}
def build_headers():
    
    token=fetch_tenant_token(bitapp_auth)
    #token='u-3atGsodiN9391_BlvmfeZh1kkwt5hlQxo0G0hgew2Ijc'
    headers = {
        'Authorization':
        'Bearer ' + token,  # your access token
        #'Content-Type': 'application/json; charset=utf-8'
    }
    return headers

In [ ]:
def download(url):
    url=url
    headers=build_headers()
    payload = ''
    #print(headers)
    file_token='bascnzsAjUXwr4gUkaIpqi8BBaf'
    table_token='tbl0Ko7cxJaGwUnh'
    record_id="recV96vgmB"
    save_path=os.path.join(r'E:\wangzhilin\QuantumGalaxy\pics',file_token)
    save_path=os.path.json(r'E:\wangzhilin\QuantumGalaxy\server_new\blueprints\pic\static',file_token)
    #save_file=os.path.join(save_path,record_id)
    res=  requests.request("GET", url, headers=headers, data=payload,stream=True)
    if res.headers['Content-Type'] in ['image/gif', 'image/png', 'image/jpeg', 'image/bmp', 'image/webp']:
        post=res.headers['Content-Type'].split('/')[1]
        #post=res.headers['Content-Disposition'].split('.')[-1]
        name=record_id+'.'+post
        save_file=os.path.join(save_path,name)
        #return res.headers
        if not os.path.exists(save_path):
            os.makedirs(save_path)
        with  open(save_file, "wb") as f:
            # 下载文件大小
            total_size = int(res.headers["content-length"])
            content_size = 0
            for content in res.iter_content(chunk_size=1024):
                f.write(content)
                content_size += len(content)

        data = {"recv_size": content_size, "total_size": total_size}
        if content_size == total_size:
            return ("接收成功")
        return ("接收失败")
    else:return('wrong type, got "%s"'%res.headers['Content-Type'])

r=download(r"https://open.feishu.cn/open-apis/drive/v1/medias/boxcnyh7dn98QVYTIGQZI4hPCOc/download?extra=%7B%22bitablePerm%22%3A%7B%22tableId%22%3A%22tbl0Ko7cxJaGwUnh%22%2C%22rev%22%3A77%7D%7D")
r

In [ ]:
def upload(file):
    UPDATE_PACKAGE_PATH = '/tools/package/'
    out = os.path.join(UPDATE_PACKAGE_PATH, params.get("filename"))
    if not os.path.isfile(out):
        return None
    # 文件大小
    fsize = os.path.getsize(out)

    def send_file_fp():
        store_path = out
        send_size = 0
        with open(store_path, "rb") as target_file:
            while 1:
                data = target_file.read(2 * 1024 * 1024)  # 每次读取2M
                if not data:
                    break
                yield data
    #content_type='multipart/form-data; boundary=something'
    content_type='image/'+name.split('.')[1]
    response = Response(send_file_fp(), )
    response.headers["Content-disposition"] = 'attachment; filename={}'.format(params.get("filename"))
    response.headers["Content-length"] = fsize
    return response


In [ ]:
def find_file(token,id):
    import os
    root_dir=r'E:\wangzhilin\QuantumGalaxy\server_new\blueprints\pic\static'
    file_dir=os.path.join(root_dir,token)
    files=os.listdir(file_dir)
    file_name=None
    for file in files:
        name=os.path.splitext(file)
        if name[0]==id:
            file_name=name[0]+name[1]
    return  file_dir,file_name
   
file_token='bascnzsAjUXwr4gUkaIpqi8BBaf'
table_token='tbl0Ko7cxJaGwUnh'
record_id="recV96vgmB"
r=find_file(file_token,record_id)

In [ ]:
from flask import Response
import os
def test():
    file_token='bascnzsAjUXwr4gUkaIpqi8BBaf'
    table_token='tbl0Ko7cxJaGwUnh'
    record_id="recV96vgmB"
    file_dir,file_name=find_file(file_token,record_id)
    if file_name:
        out=os.path.join(file_dir,file_name)
        fsize = os.path.getsize(out)

        def send_file_fp():
            store_path = out
            send_size = 0
            with open(store_path, "rb") as target_file:
                while 1:
                    data = target_file.read(2 * 1024 * 1024)  # 每次读取2M
                    if not data:
                        break
                    yield data
        #content_type='multipart/form-data; boundary=something'
        content_type='image/'+out.split('.')[1]
        response = Response(send_file_fp(),content_type=content_type )
        response.headers["Content-disposition"] = 'attachment; filename={}'.format(file_name)
        response.headers["Content-length"] = fsize
        return response
r=test()
r

In [ ]:
r.headers

In [ ]:
r.content

In [ ]:
imagebyte=r.content

In [ ]:
import cv2
import numpy as np

def read2byte(path):
    '''
    图片转二进制
    path：图片路径
    byte_data：二进制数据
    '''
    with open(path,"rb") as f:
        byte_data = f.read()
    return byte_data

def byte2numpy(byte_data):
    '''
    byte转numpy矩阵/cv格式
    byte_data：二进制数据
    image : numpy矩阵/cv格式图片
    '''
    image = np.asarray(bytearray(byte_data), dtype="uint8")
    image = cv2.imdecode(image, cv2.IMREAD_COLOR)
    return image


整合部分
对各个表格data进行整合（排除其中error的部分？）